In [ ]:
# preamble
import torch
import matplotlib.pyplot as plt
from data_generator import DataGenerator
from c_model import CModel
from x_model import XModel
from helpers import load_model
from main import compute_prior, probs
from inconsistency import inconsistency

suffixes = ['0', '1', '2', '3', '4', '5', 'cs']
descr =    ['chain length = 1.25', 
            'chain length = 2',
            'chain length = 5',
            'chain length = 10',
            'chain length = 20',
            'chain length = 50',
            'consistent']

nfiles = len(suffixes)
names = ['./logs/log-' + s + '.txt' for s in suffixes]

# default colors from T10-palette
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

plt.rcParams['text.usetex'] = True
plt.rcParams.update({'font.size': 24})
plt.rcParams["figure.figsize"] = (20,6)
fig, ax = plt.subplots(1, 1)

x = torch.linspace(-2, 2, 1000)

leg = []
data_generator = DataGenerator(10)
ax.plot(x, data_generator.p, color='black');
leg += ['data distribution']

# models to plot
to_viz = [0,2,4]
i = 0
for v in to_viz:
    c_model = CModel(0)
    x_model = XModel(0, False)
    load_model(c_model, './models/c_' + suffixes[v] + '.pt', strict=True, load_optimizer=False)
    load_model(x_model, './models/x_' + suffixes[v] + '.pt', strict=True, load_optimizer=False)
    cprobs = compute_prior(c_model, x_model, nsamples=100000, nit=100)
    xprobs = probs(x_model, cprobs)
    ax.plot(x, xprobs, color=colors[i]);
    leg += [descr[v]]
    i += 1

# best consistent
c_model = CModel(0)
x_model = XModel(0, True)
load_model(c_model, './models/c_' + suffixes[-1] + '.pt', strict=True, load_optimizer=False)
load_model(x_model, './models/x_' + suffixes[-1] + '.pt', strict=True, load_optimizer=False)
cprobs = compute_prior(c_model, x_model, nsamples=100000, nit=100)
xprobs = probs(x_model, cprobs)
ax.plot(x, xprobs, '--', color='black');
leg += [descr[-1]]

ax.grid(True);
ax.set_xlim((-2,1.5))
ax.legend(leg, loc='upper left');


In [ ]:
kl = []
inc = []
skl = []
for i in range(nfiles):
    c_model = CModel(0)
    x_model = XModel(0, (i==nfiles-1))
    load_model(c_model, './models/c_' + suffixes[i] + '.pt', strict=True, load_optimizer=False)
    load_model(x_model, './models/x_' + suffixes[i] + '.pt', strict=True, load_optimizer=False)
    cprobs = compute_prior(c_model, x_model, nsamples=100000, nit=100)
    xprobs = probs(x_model, cprobs)
    kl += [data_generator.kl(xprobs).numpy()]
    inc += [inconsistency(c_model, x_model, 100000, nit_warmup=100, nit_acc=1000)]

    # joint probability p(k) * p(x|k)
    mu0, sigma0 = x_model.get_ms(torch.zeros([1,1]))
    mu1, sigma1 = x_model.get_ms(torch.ones([1,1]))
    p0 = torch.log(sigma0) + ((x - mu0)**2) / (2*(sigma0**2))
    p0 = (1 - cprobs) * torch.exp(-p0).flatten()
    p1 = torch.log(sigma1) + ((x - mu1)**2) / (2*(sigma1**2))
    p1 = cprobs * torch.exp(-p1).flatten()
    pkx = torch.stack((p0, p1))
    pkx = pkx * (1 - 1e-3) + 0.5 * 1e-6
    pkx = pkx / pkx.sum()
    lpkx = torch.log(pkx)

    # joint probability p(x) * p(k|x)
    with torch.no_grad():
        kkk = c_model.net(x.unsqueeze(-1)).flatten().sigmoid().detach()
    pk0 = xprobs * (1 - kkk)
    pk1 = xprobs * kkk
    pxk = torch.stack((pk0, pk1))
    pxk = pxk * (1 - 1e-3) + 0.5 * 1e-6
    pxk = pxk / pxk.sum()
    lpxk = torch.log(pxk)

    # symmetrized kl
    skl += [(pkx * (lpkx - lpxk) + pxk * (lpxk - lpkx)).sum()]


In [ ]:
nch = [1.25, 2, 5, 10, 20, 50]

plt.rcParams['text.usetex'] = True
plt.rcParams.update({'font.size': 24})
plt.rcParams["figure.figsize"] = (20,5.5)
fig, ax = plt.subplots(1, 2)

ax[0].plot(nch, kl[:-1], color=colors[0], linewidth=3);
ax[0].plot(nch, [kl[-1]] * len(nch), '--', color='black', linewidth=3);
ax[0].grid();
ax[0].set_xlabel('chain length');
ax[0].set_ylabel('$D_{KL}(p^*(x)\; ||\; p(x)) $');
ax[0].legend(['inconsistent', 'consistent'])

ax[1].plot(nch, skl[:-1], color=colors[0], linewidth=3);
ax[1].plot(nch, [skl[-1]] * len(nch), '--', color='black', linewidth=3);
ax[1].grid();
ax[1].set_xlabel('chain length');
ax[1].set_ylabel('Inconsistency');
ax[1].legend(['inconsistent', 'consistent']);
